# Report — MVTec AD Anomaly Detection

Questo notebook raccoglie i risultati del rilevamento di anomalie tramite un autoencoder convoluzionale denoising (denoising conv-AE), addestrato separatamente per ciascuna categoria di prodotto del dataset MVTec AD con una loss combinata SSIM + MSE, e ne confronta le prestazioni (AUROC a livello di immagine e di pixel) tra le 15 categorie e tra due varianti di ablazione. Per il design completo vedere `docs/design.md`.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

sys.path.insert(0, str(Path.cwd().parent))

from anomaly_ae.ablation import ABLATION_CATEGORIES
from anomaly_ae.report import ablation_comparison_dataframe, load_metrics, metrics_to_dataframe
from data_classes.mvtec_dataset import CATEGORIES

OUTPUT_ROOT = Path.cwd().parent / "outputs"

## 1. Confronto tra le 15 categorie

In [ ]:
main_metrics = load_metrics(OUTPUT_ROOT, CATEGORIES)
main_df = metrics_to_dataframe(main_metrics)
main_df[["image_level_auroc", "pixel_level_auroc"]].plot.bar(figsize=(12, 4))
main_df

### Ispezione qualitativa

Griglie `originale | ricostruzione | mappa di anomalia | maschera reale` prodotte da
`python test.py` per alcune categorie rappresentative: una texture ripetitiva
(`carpet`), un oggetto rigido ben posizionato (`bottle`) e un oggetto piccolo e
difficile (`screw`).

In [ ]:
for category in ["bottle", "carpet", "screw"]:
    sample_paths = sorted((OUTPUT_ROOT / category).glob("sample_*.png"))
    for path in sample_paths[:2]:
        print(f"{category}: {path.name}")
        display(Image(filename=str(path)))

### Curve di training

Loss di training e di validation per epoca (`history.csv`), utile per verificare
convergenza ed effetto dell'early stopping sulle stesse categorie.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, category in zip(axes, ["bottle", "carpet", "screw"]):
    history = pd.read_csv(OUTPUT_ROOT / category / "history.csv")
    ax.plot(history["epoch"], history["train_loss"], label="train")
    ax.plot(history["epoch"], history["val_loss"], label="val")
    ax.set_title(category)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.legend()
plt.tight_layout()

### Decisione concreta alla soglia

L'AUROC è una metrica continua, indipendente dalla soglia. Usando gli score
per-immagine e la soglia (95° percentile degli score sul train set `good`) ora
salvati in `metrics.json`, si può mostrare la decisione binaria normale/anomalo
effettivamente presa dal sistema: di seguito l'accuratezza per categoria a quella
soglia.

In [ ]:
rows = []
for category, m in main_metrics.items():
    scores = m["image_scores"]
    labels = m["image_labels"]
    threshold = m["threshold"]
    predictions = [1 if s > threshold else 0 for s in scores]
    correct = sum(p == l for p, l in zip(predictions, labels))
    rows.append({"category": category, "accuracy_at_threshold": correct / len(labels)})
pd.DataFrame(rows).set_index("category")

## 2. Ablation — loss SSIM+MSE vs solo MSE

In [ ]:
mse_only_metrics = load_metrics(OUTPUT_ROOT / "ablation_mse_only", ABLATION_CATEGORIES)
main_subset = {c: main_metrics[c] for c in ABLATION_CATEGORIES}
loss_ablation_df = ablation_comparison_dataframe(main_subset, mse_only_metrics, "mse_only")
loss_ablation_df[["image_auroc_main", "image_auroc_mse_only"]].plot.bar(figsize=(10, 4))
loss_ablation_df

## 3. Ablation — denoising vs senza rumore

In [ ]:
no_denoising_metrics = load_metrics(OUTPUT_ROOT / "ablation_no_denoising", ABLATION_CATEGORIES)
denoising_ablation_df = ablation_comparison_dataframe(main_subset, no_denoising_metrics, "no_denoising")
denoising_ablation_df[["image_auroc_main", "image_auroc_no_denoising"]].plot.bar(figsize=(10, 4))
denoising_ablation_df

## 4. Discussione finale

_Da completare dopo aver osservato i risultati reali:_

- Quale combinazione di scelte (loss, denoising) funziona meglio, e su quali tipologie di categoria (texture vs oggetti rigidi)?